In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split

# 1. LOAD DATA
historical = pd.read_csv("Historical Data.csv")
realtime = pd.read_csv("Real-Time Data.csv")

# Create target variable
historical["Risk_Flag"] = historical["Clearance_Status"].apply(
    lambda x: 0 if x == "Clear" else 1
)


# 2. BASE FEATURE ENGINEERING (Row-level, no data leakage risk)
def apply_base_features(df):
    df = df.copy()
    # Weight differences
    df["Weight_Diff"] = abs(df["Declared_Weight"] - df["Measured_Weight"])
    df["Weight_Diff_Percent"] = (df["Weight_Diff"] / df["Declared_Weight"]) * 100
    df["Value_per_KG"] = df["Declared_Value"] / df["Declared_Weight"]

    # Trade routes
    df["Trade_Route"] = (
        df["Origin_Country"].astype(str) + "_" + df["Destination_Country"].astype(str)
    )

    # Date/Time Parsing
    df["Declaration_Date"] = pd.to_datetime(df["Declaration_Date (YYYY-MM-DD)"])
    df["Month"] = df["Declaration_Date"].dt.month
    df["Day"] = df["Declaration_Date"].dt.day

    df["Declaration_Time"] = pd.to_datetime(
        df["Declaration_Time"], format="%H:%M:%S", errors="coerce"
    )
    df["Hour"] = df["Declaration_Time"].dt.hour.fillna(0).astype(int)
    df["Night_Declaration"] = df["Hour"].apply(lambda x: 1 if x < 6 else 0)

    # Dwell Time risk
    df["High_Dwell_Risk"] = df["Dwell_Time_Hours"].apply(
        lambda x: 1 if x > 48 else 0
    )

    # Ensure HS_Code is a string
    df["HS_Code"] = df["HS_Code"].astype(str)
    return df


historical = apply_base_features(historical)
realtime = apply_base_features(realtime)


# 3. SAFE CATEGORICAL ENCODING FUNCTION
categorical_cols = [
    "Trade_Regime (Import / Export / Transit)",
    "Origin_Country",
    "Destination_Country",
    "Destination_Port",
    "Shipping_Line",
    "HS_Code",
]

# Create safe map from historical data
for col in categorical_cols:
    historical[col] = historical[col].astype(str)
    realtime[col] = realtime[col].astype(str)

    # Build category map based on historical data
    unique_cats = historical[col].unique()
    cat_map = {cat: idx for idx, cat in enumerate(unique_cats)}

    historical[col] = historical[col].map(cat_map)
    realtime[col] = realtime[col].map(cat_map).fillna(-1).astype(int)


# 4. ADVANCED AGGREGATION FUNCTION (Prevents code repetition)
def build_aggregated_features(source_df, target_df, is_validation=False):
    target_df = target_df.copy()

    # Counts & Frequencies
    target_df["Importer_Shipment_Count"] = target_df["Importer_ID"].map(
        source_df.groupby("Importer_ID").size()
    ).fillna(0)
    target_df["Exporter_Shipment_Count"] = target_df["Exporter_ID"].map(
        source_df.groupby("Exporter_ID").size()
    ).fillna(0)
    target_df["Route_Frequency"] = target_df["Trade_Route"].map(
        source_df.groupby("Trade_Route")["Container_ID"].count()
    ).fillna(0)

    # Deviations & Means
    target_df["Importer_Avg_Weight_Diff"] = target_df["Importer_ID"].map(
        source_df.groupby("Importer_ID")["Weight_Diff"].mean()
    ).fillna(0)
    target_df["Exporter_Avg_Weight_Diff"] = target_df["Exporter_ID"].map(
        source_df.groupby("Exporter_ID")["Weight_Diff"].mean()
    ).fillna(0)

    importer_avg_value = source_df.groupby("Importer_ID")["Declared_Value"].mean()
    target_df["Importer_Value_Deviation"] = abs(
        target_df["Declared_Value"] - target_df["Importer_ID"].map(importer_avg_value)
    ).fillna(0)

    # Risk Scores (Target Encodings)
    global_risk = source_df["Risk_Flag"].mean()

    target_df["Importer_Risk_Score"] = target_df["Importer_ID"].map(
        source_df.groupby("Importer_ID")["Risk_Flag"].mean()
    ).fillna(0)
    target_df["Exporter_Risk_Score"] = target_df["Exporter_ID"].map(
        source_df.groupby("Exporter_ID")["Risk_Flag"].mean()
    ).fillna(0)
    target_df["Shipping_Risk"] = target_df["Shipping_Line"].map(
        source_df.groupby("Shipping_Line")["Risk_Flag"].mean()
    ).fillna(0)
    target_df["Port_Risk"] = target_df["Destination_Port"].map(
        source_df.groupby("Destination_Port")["Risk_Flag"].mean()
    ).fillna(global_risk)
    target_df["Trade_Route_Risk"] = target_df["Trade_Route"].map(
        source_df.groupby("Trade_Route")["Risk_Flag"].mean()
    ).fillna(0)
    target_df["HS_Code_Risk"] = target_df["HS_Code"].map(
        source_df.groupby("HS_Code")["Risk_Flag"].mean()
    ).fillna(0)

    return target_df


# Define complete list of features for the model
features = [
    "Weight_Diff",
    "Weight_Diff_Percent",
    "Value_per_KG",
    "Importer_Shipment_Count",
    "Exporter_Shipment_Count",
    "Importer_Avg_Weight_Diff",
    "Exporter_Avg_Weight_Diff",
    "Importer_Risk_Score",
    "Exporter_Risk_Score",
    "Shipping_Risk",
    "Port_Risk",
    "Trade_Route_Risk",
    "HS_Code_Risk",
    "Route_Frequency",
    "Dwell_Time_Hours",
    "High_Dwell_Risk",
    "Trade_Regime (Import / Export / Transit)",
    "Origin_Country",
    "Destination_Country",
    "Destination_Port",
    "Shipping_Line",
    "HS_Code",
    "Month",
    "Day",
    "Hour",
    "Night_Declaration",
    "Importer_Value_Deviation",
]

# --- PHASE A: VALIDATION (Strictly No Leakage) ---
print("--- Evaluating Model Performance via Validation Split ---")
train_split, val_split = train_test_split(
    historical, test_size=0.2, random_state=42, stratify=historical["Risk_Flag"]
)

# Map aggregations using ONLY train_split statistics
X_train_val = build_aggregated_features(train_split, train_split)
X_val_val = build_aggregated_features(train_split, val_split)

# Clean infinite/NaN values safely
X_train_val_f = X_train_val[features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val_val_f = X_val_val[features].replace([np.inf, -np.inf], np.nan).fillna(0)

y_train_val = X_train_val["Risk_Flag"].fillna(0)
y_val_val = X_val_val["Risk_Flag"].fillna(0)

# Validate model
val_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42
)
val_model.fit(X_train_val_f, y_train_val)
y_val_pred = val_model.predict(X_val_val_f)

print("Macro F1:", f1_score(y_val_val, y_val_pred, average="macro"))
print("Weighted F1:", f1_score(y_val_val, y_val_pred, average="weighted"))
print("Recall Critical:", recall_score(y_val_val, y_val_pred, pos_label=1))
print("\nConfusion Matrix:\n", confusion_matrix(y_val_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val_val, y_val_pred))


# --- PHASE B: PRODUCTION PRODUCTION (Fitting on Full Data for Real-Time Inference) ---
print("\n--- Training Final Models on Full Historical Dataset ---")
historical_proc = build_aggregated_features(historical, historical)
realtime_proc = build_aggregated_features(historical, realtime)

X_full = historical_proc[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_full = historical_proc["Risk_Flag"].fillna(0)
X_test = realtime_proc[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Train Production Random Forest
final_rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42
)
final_rf.fit(X_full, y_full)

# Train Production Isolation Forest
final_iso = IsolationForest(contamination=0.05, random_state=42)
final_iso.fit(X_full)

# Predict on Realtime Data
realtime_proc["Predicted_Risk"] = final_rf.predict(X_test)
realtime_proc["Risk_Probability"] = final_rf.predict_proba(X_test)[:, 1]

# Outlier Detection
realtime_proc["Anomaly_Score"] = final_iso.predict(X_test)
realtime_proc["Anomaly_Flag"] = realtime_proc["Anomaly_Score"].apply(
    lambda x: 1 if x == -1 else 0
)

# Map Dynamic Risk Level
realtime_proc["Risk_Level"] = realtime_proc["Risk_Probability"].apply(
    lambda x: "High Risk" if x > 0.7 else "Medium Risk" if x > 0.4 else "Low Risk"
)

# Unified Final Risk Decision Matrix
realtime_proc["Final_Risk"] = realtime_proc.apply(
    lambda row: (
        "High Risk"
        if row["Risk_Level"] == "High Risk" or row["Anomaly_Flag"] == 1
        else row["Risk_Level"]
    ),
    axis=1,
)


# 5. RISK EXPLANATION ENGINE
def explain_risk(row):
    reasons = []
    if row["Weight_Diff_Percent"] > 20:
        reasons.append("Large weight difference detected")
    if row["Dwell_Time_Hours"] > 48:
        reasons.append("Shipment stayed unusually long at port")
    if row["Importer_Risk_Score"] > 0.5:
        reasons.append("Importer has previous risky shipment history")
    if row["Exporter_Risk_Score"] > 0.5:
        reasons.append("Exporter has previous risky shipment history")
    if row["HS_Code_Risk"] > 0.5:
        reasons.append("Product category historically high risk")
    if row["Trade_Route_Risk"] > 0.5:
        reasons.append("Trade route historically high risk")

    return " | ".join(reasons) if reasons else "No major risk indicators detected"


realtime_proc["Risk_Explanation"] = realtime_proc.apply(explain_risk, axis=1)

# Extract and Save Important Features
feature_importance = pd.DataFrame(
    {"Feature": features, "Importance": final_rf.feature_importances_}
).sort_values(by="Importance", ascending=False)


# 6. SAVE RESULTS
result_summary = realtime_proc[
    [
        "Importer_ID",
        "Exporter_ID",
        "HS_Code",
        "Risk_Probability",
        "Risk_Level",
        "Final_Risk",
        "Risk_Explanation",
    ]
]

print("\nReal-Time Risk Predictions Preview:")
print(result_summary.head())

result_summary.to_csv("shipment_risk_results.csv", index=False)
feature_importance.to_csv("feature_importance.csv", index=False)
realtime_proc.to_csv("realtime_with_risk_predictions.csv", index=False)
print("\nProcessing complete! All artifacts saved successfully.")

--- Evaluating Model Performance via Validation Split ---
Macro F1: 0.9780086343717974
Weighted F1: 0.9850089102674122
Recall Critical: 0.9841269841269841

Confusion Matrix:
 [[8343  126]
 [  37 2294]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99      8469
           1       0.95      0.98      0.97      2331

    accuracy                           0.98     10800
   macro avg       0.97      0.98      0.98     10800
weighted avg       0.99      0.98      0.99     10800


--- Training Final Models on Full Historical Dataset ---

Real-Time Risk Predictions Preview:
  Importer_ID Exporter_ID  HS_Code  Risk_Probability Risk_Level Final_Risk  \
0     Y3OM027     FQKH1PD      210          0.014551   Low Risk   Low Risk   
1     QBIH1EB     543HRUP      110          0.081783   Low Risk   Low Risk   
2     KEABR8T     69Z0KLV      263          0.022917   Low Risk   Low Risk   
3     P0E1D1R     LFV52UK       26     

In [3]:
import joblib

artifacts = {
    "rf_model": final_rf,
    "iso_model": final_iso,
    "features": features
}

joblib.dump(artifacts, "smartcontainer_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [4]:
import os
os.getcwd()

'C:\\Users\\Aarzoo\\AppData\\Local\\Programs\\Python\\Python313\\Scripts\\smartcontainer_risk_prediction'